# 70 — Train BGE-M3 bi-encoder (Stage A) — end-to-end

Single-path notebook. Fine-tunes a pretrained dense retriever on a fraction
of HF train,
trains on a user-disjoint train/val split, pushes the merged model
to HF Hub, re-embeds the ~47K-track catalog, and reports standalone
dev nDCG@20.

## Quick start

Set `SKIP_MINING = True` (default) in cell 1 to reuse an existing
`TRIPLES_JSONL` and run cells 2-7 end-to-end. Set `SKIP_MINING = False`
to force a fresh HN mine.


1. Set the parameters in **cell 1 (CONFIG)** — `MAX_INPUT_ROWS`, `EPOCHS`,
   `PER_DEVICE_BATCH_SIZE`, etc. — to control the run size.
2. Run cells 2-7 sequentially. Open cell 5 (TensorBoard) in parallel with cell 4 (train).

## What we're verifying

1. **train/val nDCG alignment** — `train/ndcg_inbatch` and `val/ndcg`
   curves should track each other (gap < ~0.05 sustained). Gap > 0.05 = leak.
2. **honest retrieval lift** — `val/full_catalog_ndcg_at_20` (vs the real ~47K
   catalog) should climb above ~0.05-0.08 by end of epoch 1.
3. **HF train/test disjointness** — preflight in cell 7 refuses to score
   if HF train and test splits share user_ids or session_ids.

## Warm-start across sessions

Per-epoch LoRA checkpoints save to `TRAIN_OUTPUT_DIR/checkpoint_epoch_{1,2,3}/` on
Drive — they survive Colab session restarts. To resume training in a future session,
set `RESUME_FROM` in cell 1 to one of those paths and run cells 1–7.

## Recent patches (2026-05-22)

- **`BGE_MODEL` backbone knob** in CONFIG. Current default `BAAI/bge-base-en-v1.5`
  (110M, English-only) — 5× smaller than BGE-M3, allows much larger batches.
  Set to `BAAI/bge-m3` to revert.


After BlindA nDCG regressed (0.06 baseline → 0.05 fine-tuned), a systematic-
debugging audit found TWO logical bugs. Both are fixed in this commit:

- **Mining filter selection bias** — `percpos@0.95` skipped 60% of queries
  (the hard ones), biasing the training distribution toward easy queries.
  Fix: new CONFIG knob `MINING_STRATEGY = 'percpos' | 'simans'`. Switch to
  SimANS to eliminate the filter and use 100% of queries.
- **Catalog vs [HISTORY] format mismatch** — pos/neg used `format_track_text`
  (5 fields, pipe-separated, original case) while [HISTORY] music-turn
  references used `id_to_metadata` (4 fields, comma, lowercased). The
  encoder had to learn two representations of every track. Fix: builder
  and cell 6 now both use `id_to_metadata` format → everything aligned.

**To take effect**, set `MINING_STRATEGY = 'simans'` in CONFIG, then re-mine
(cell 3 with SKIP_MINING=False) + re-embed (cell 6) + retrain (cell 4).

## §6.5 amendment knobs active in this run

- `--split-key user_id` — user-disjoint train/val (every session of a given user lives in one partition).
- `UserDisjointBatchSampler` — distinct users per batch; without-replacement; heap-greedy load-balanced.
- In-batch InfoNCE with false-positive collision masking (same-pos-tid + cross-pos-into-neg-slot; RocketQAv2 / BGE-M3 §3.3).
- Per-row `K_data = N_NEGATIVES` mined negs; rows below threshold are dropped (no upsampling).


In [ ]:
# 1) CONFIG — all run parameters in one place. Edit then run cells 2-7.

# CRITICAL: set GPU memory env vars BEFORE any imports that pull JAX/TF.
# Colab's `datasets` + `transformers` transitively import JAX, which
# preallocates 90% of GPU memory by default (~85 GB out of 95 GB on
# Blackwell). That blocks the training subprocess in cell 4 from getting
# enough VRAM → OOM. These vars force growth-only mode. MUST be set in the
# kernel BEFORE the first datasets/transformers/jax import — otherwise no
# effect (you'd need to restart the runtime).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- Branch + identity --------------------------------------------------
BRANCH       = 'fresh-model'
HUB_USER     = 'OrRim123'
RUN_NAME     = 'bge-base-en-music-v1'  # used as suffix everywhere; matches production wRRF factory
                                  # IMPORTANT: also change RUN_NAME when changing BGE_MODEL below,
                                  # so the Hub repo name reflects the actual model architecture.

BGE_MODEL    = 'BAAI/bge-base-en-v1.5'  # Pre-trained encoder backbone. Tested options:
                                  #   'BAAI/bge-m3'             — 567M params, 1024-d, multilingual (default)
                                  #   'BAAI/bge-base-en-v1.5'   — 110M params,  768-d, English-only (5× smaller;
                                  #     allows MUCH larger batch sizes — try MINING_BATCH_SIZE=256,
                                  #     PER_DEVICE_BATCH_SIZE=64, GRAD_ACCUM_STEPS=1 for ~3× faster training)
                                  #   'BAAI/bge-large-en-v1.5'  — 335M params, 1024-d, English-only

HUB_REPO_OVERRIDE = 'OrRim123/recsys2026-bge-m3-music-v1'  # Reuses the existing v1 Hub repo
                                  # (was '' = auto-derive). Set to '' to revert. Use cases:
                                  #   - Reuse an existing repo you have write access to (e.g. v1) instead
                                  #     of creating a new one. Train script will overwrite that repo's
                                  #     contents with the newly-trained model.
                                  #   - Avoid the 'repo doesn't exist yet' 404 when running cell 6/7
                                  #     against an existing Hub repo before re-training.
                                  # Format: '<HUB_USER>/recsys2026-<some-name>' (script appends '-merged')
                                  # Example: 'OrRim123/recsys2026-bge-m3-music-v1' → pushes to
                                  #          'OrRim123/recsys2026-bge-m3-music-v1-merged' (existing repo)
                                  # Empty = use auto-derived 'OrRim123/recsys2026-{RUN_NAME}'.

# --- Data scope ---------------------------------------------------------
SKIP_MINING    = False          # True = use existing TRIPLES_JSONL if present;
                                # False = always re-mine.
MAX_INPUT_ROWS = 0              # 0 = use ALL ~121K HF train turns; set 12000 for quick-iter (~10%)
DEV_EVAL_ROWS  = 8000           # # of HF test rows scored in cell 7 (use full HF test split)

# --- HN mining ----------------------------------------------------------
MINING_STRATEGY   = 'simans'    # 'percpos' (filter; ~60% skip rate at threshold=0.95) or
                                # 'simans'  (Gaussian-weighted, no skip; uses 100% of queries).
                                # SimANS recommended when val/dev/BlindA show train-data
                                # selection bias (high val nDCG but low dev/BlindA nDCG).
PERCPOS_THRESHOLD = 0.95        # candidates KEPT if score < threshold * pos_score (percpos only)
POOL_SIZE         = 1000        # top-K candidates considered per query
N_NEGATIVES       = 15          # negs per row (rows below this are DROPPED at train time)
MINING_BATCH_SIZE = 256         # 4× larger than BGE-M3 default (bge-base is 5× smaller)
SIMANS_A          = 0.1         # SimANS Gaussian peak offset (s_pos - a); used when MINING_STRATEGY='simans'
SIMANS_B          = 0.05        # SimANS Gaussian spread; smaller = narrower peak around target

# --- Training -----------------------------------------------------------
EPOCHS                     = 1        # 3 for production; 1 for quick-iter sanity (start here)
SEED                       = 42       # master seed (data split, samplers, LoRA init)
PER_DEVICE_BATCH_SIZE      = 64       # 4× larger than BGE-M3 default (bge-base smaller)
GRAD_ACCUM_STEPS           = 1        # effective batch = PER_DEVICE_BATCH_SIZE (64) directly
LR                         = 5e-6
TEMPERATURE                = 0.05
LORA_RANK                  = 64
LORA_ALPHA                 = 128
QUERY_MAX_LEN              = 384
PASSAGE_MAX_LEN            = 192
SPLIT_KEY                  = 'user_id'   # 'user_id' | 'session_id' | 'row'
VAL_FRACTION               = 0.10
LOGGING_STEPS              = 25       # log every N opt-steps; 5 for quick-iter
VAL_EVERY_N_STEPS          = 100      # val pass every N opt-steps; 10 for quick-iter
VAL_FULL_CATALOG_EVERY_N   = 200      # full-catalog (~50 sec each) every N opt-steps; 50 for quick-iter
CHECKPOINT_EVERY_N_EPOCHS  = 1        # save adapter per epoch (~30 MB each) for warm-start/revert
RESUME_FROM                = ''       # '' = train from scratch; otherwise absolute path to a checkpoint_epoch_N/
                                      # directory under TRAIN_OUTPUT_DIR. Optimizer/scheduler restart fresh
                                      # (script default); --epochs counts ADDITIONAL epochs from the checkpoint.
GRADIENT_CHECKPOINTING     = True
TENSORBOARD_PORT           = 6006

# --- Derived paths (do not edit usually) --------------------------------
HUB_REPO         = HUB_REPO_OVERRIDE if HUB_REPO_OVERRIDE else f'{HUB_USER}/recsys2026-{RUN_NAME}'
HUB_REPO_MERGED  = f'{HUB_REPO}-merged'
EMBED_LABEL      = f'{RUN_NAME}-merged'
TRIPLES_JSONL    = f'experiments/cache/retrieval_v2/triples_{RUN_NAME.replace("-","_")}.jsonl'
# Training artifacts (LoRA adapter + checkpoint_epoch_*/ + runs/) live on
# Drive so per-epoch checkpoints SURVIVE Colab session restarts → enables
# cross-session warm-start via RESUME_FROM above.
TRAIN_OUTPUT_DIR = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/training/{RUN_NAME.replace("-","_")}'
DRIVE_RESULTS    = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results/{RUN_NAME.replace("-","_")}'
DRIVE_LOG_HN     = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_hn_log.txt'
DRIVE_LOG_TRAIN  = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_train_log.txt'
CACHE_ROOT       = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
SAFE_MODEL       = HUB_REPO_MERGED.replace('/', '_')
CATALOG_OUT_DIR  = f'{CACHE_ROOT}/{SAFE_MODEL}/{EMBED_LABEL}'

print('Config loaded:')
for k in ('BRANCH','BGE_MODEL','HUB_REPO_OVERRIDE','HUB_REPO','HUB_REPO_MERGED','TRIPLES_JSONL','TRAIN_OUTPUT_DIR',
         'SKIP_MINING','RESUME_FROM','SEED','MAX_INPUT_ROWS','EPOCHS','PER_DEVICE_BATCH_SIZE','GRAD_ACCUM_STEPS',
         'CHECKPOINT_EVERY_N_EPOCHS','N_NEGATIVES','MINING_STRATEGY','PERCPOS_THRESHOLD','SPLIT_KEY','VAL_FRACTION'):
    print(f'  {k} = {globals()[k]!r}')


In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s' 'torchao>=0.16' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'


In [ ]:
# 3) HN mine — produces TRIPLES_JSONL from HF train.
# Honors SKIP_MINING: if the JSONL already exists on Drive and the flag
# is True, this cell skips the ~25-85 min mine and just confirms the
# file's there. Set SKIP_MINING = False in cell 1 to force a re-mine.
#
# PATCH 4 (2026-05-22): the builder now writes pos/neg in id_to_metadata
# format (matching cell 6's catalog re-embed + the [HISTORY] expansion).
# Old triples_*.jsonl files mined before this patch have pos/neg in the
# legacy 5-field format — set SKIP_MINING=False to re-mine, OR keep
# SKIP_MINING=True if you want to compare against the old format.
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

_existing_triples = os.path.exists(TRIPLES_JSONL)
if SKIP_MINING and _existing_triples:
    print(f'[mine] SKIP_MINING=True and {TRIPLES_JSONL} exists — skipping mine.')
    !wc -l {TRIPLES_JSONL}
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('pos[0] sample:', r['pos'][0][:200]); print('keys:', sorted(r.keys()))"
else:
    if SKIP_MINING and not _existing_triples:
        print(f'[mine] SKIP_MINING=True but {TRIPLES_JSONL} not found — mining anyway.')
    !rm -f {TRIPLES_JSONL}
    _max_rows_flag = f'--max-rows {MAX_INPUT_ROWS}' if MAX_INPUT_ROWS > 0 else ''
    _strategy_flags = f'--mining-strategy {MINING_STRATEGY}'
    if MINING_STRATEGY == 'simans':
        _strategy_flags += f' --simans-a {SIMANS_A} --simans-b {SIMANS_B}'
    print(f'[mine] strategy={MINING_STRATEGY} (percpos→filtered+skip; simans→no-skip, all queries kept)')
    !cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
        --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
        --bge-m3-model {BGE_MODEL} \
        --output {TRIPLES_JSONL} \
        --query-mode bge_m3_structured \
        {_strategy_flags} \
        --percpos-threshold {PERCPOS_THRESHOLD} --pool-size {POOL_SIZE} \
        --k-negs {N_NEGATIVES} --batch-size {MINING_BATCH_SIZE} \
        {_max_rows_flag} \
        2>&1 | tee {DRIVE_LOG_HN}
    !wc -l {TRIPLES_JSONL}
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('pos[0] sample:', r['pos'][0][:200]); print('keys:', sorted(r.keys()))"


In [ ]:
# 3b) (One-shot) Expand music-turn IDs in [HISTORY]: blocks to match
# production inference (id_to_metadata format). Closes the train/eval
# parity gap on the [HISTORY] music turns.
#
# This is idempotent and FAST (~1-2 min CPU): only the `query` field is
# rewritten; all other fields (pos/neg/pos_tid/neg_tids/user_id/session_id)
# are preserved byte-for-byte.
#
# After the builder fix in commit (see history-corpus-types arg), fresh
# mines from cell 3 already emit the corrected format → this cell becomes
# a no-op (rewrites 0 rows). Kept for safety + to salvage any pre-fix
# JSONL still around on Drive.
TRIPLES_FIXED = TRIPLES_JSONL.replace('.jsonl', '_history_fixed.jsonl')
!cd /content/recsys2026 && python -u scripts/expand_history_in_triples.py \
    --input {TRIPLES_JSONL} \
    --output {TRIPLES_FIXED} \
    --track-meta-hf talkpl-ai/TalkPlayData-Challenge-Track-Metadata \
    --corpus-types track_name,artist_name,album_name
# Swap in the fixed file so cell 4 (train) picks it up automatically.
!mv {TRIPLES_FIXED} {TRIPLES_JSONL}
# Confirm one row's [HISTORY] now contains `A: track_id: <id>, track_name:...`.
!python3 -c "import json; r=json.loads(open('{TRIPLES_JSONL}').readline()); print(r['query'][:600])"


In [ ]:
# 3c) DIAGNOSTIC — s_pos distribution: TRAIN-mined vs DEV.
#
# Why: percpos mining filters out queries where the gold doesn't stand
# out from a 1000-candidate pool. The 60% skip rate at threshold=0.95
# creates a curated training subset biased toward HIGH s_pos queries.
# Dev (HF test) and BlindA include the FULL s_pos distribution.
# This cell quantifies that bias by computing zero-shot s_pos via BGE_MODEL for
# a sample of TRAIN-mined queries vs DEV queries.
#
# Read with: 'how much smaller is the train-mined s_pos distribution's
# tail than dev's tail?' Big gap → mining filter selection bias confirmed.
# Skippable for production runs — uncomment the early `raise` to disable.
# raise SystemExit('diagnostic disabled')
import json, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns, _format_history_music_turn

DIAG_SAMPLE = 200  # queries per side

# Load catalog + encode with zero-shot BGE-M3 (same encoder the mining used).
tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
metadata_dict = {row['track_id']: dict(row) for row in tm}
corpus_types = ['track_name', 'artist_name', 'album_name']
track_ids = [row['track_id'] for row in tm]
track_texts = [_format_history_music_turn(tid, metadata_dict, corpus_types) for tid in track_ids]
tid_to_idx = {tid: i for i, tid in enumerate(track_ids)}

# Uses BGE_MODEL (from CONFIG) so the diagnostic matches whichever encoder mining used.
# Works for any sentence-transformers-compatible model (BGE-M3, bge-base-en, etc.).
print(f'[diag] loading zero-shot encoder: {BGE_MODEL}')
model = SentenceTransformer(BGE_MODEL, device='cuda')
model.max_seq_length = 512
model = model.half()  # FP16; matches build script's mining encoder for apples-to-apples s_pos
print(f'[diag] encoding catalog ({len(track_ids):,} tracks)')
track_embs = np.asarray(model.encode(track_texts, batch_size=64,
                                     normalize_embeddings=True, convert_to_numpy=True,
                                     show_progress_bar=False), dtype=np.float32)
track_embs /= np.clip(np.linalg.norm(track_embs, axis=1, keepdims=True), 1e-9, None)

def _sample_s_pos(rows, label):
    rows = rows[:DIAG_SAMPLE]
    queries = [format_query_text(r.get('chat_history') or [], r.get('current_user_query',''),
                                  r.get('user_profile_raw'), r.get('conversation_goal'),
                                  mode='bge_m3_structured') for r in rows]
    q_embs = np.asarray(model.encode(queries, batch_size=32,
                                     normalize_embeddings=True, convert_to_numpy=True,
                                     show_progress_bar=False), dtype=np.float32)
    q_embs /= np.clip(np.linalg.norm(q_embs, axis=1, keepdims=True), 1e-9, None)
    s_pos_vals = []
    for q_emb, row in zip(q_embs, rows):
        gold = row.get('track_id')
        if gold not in tid_to_idx: continue
        s_pos = float(q_emb @ track_embs[tid_to_idx[gold]])
        s_pos_vals.append(s_pos)
    if not s_pos_vals:
        print(f'[diag] {label}: no valid s_pos values'); return None
    arr = np.asarray(s_pos_vals)
    print(f'[diag] {label}: n={len(arr)} mean={arr.mean():.3f} std={arr.std():.3f} '
          f'p10={np.percentile(arr,10):.3f} p50={np.percentile(arr,50):.3f} p90={np.percentile(arr,90):.3f}')
    return arr

print(f'\n=== TRAIN-mined sample (from {TRIPLES_JSONL}) ===')
# Match by (session_id, pos_tid) — track_id alone is ambiguous when the same
# gold track appears as the gold in multiple sessions (some of which the
# mining filter kept, others skipped). Using the tuple gives us EXACTLY
# the rows that survived mining.
train_mined_keys = set()
with open(TRIPLES_JSONL) as f:
    for line in f:
        r = json.loads(line)
        sid = r.get('session_id')
        if sid is not None:
            train_mined_keys.add((sid, r['pos_tid']))
train_ds_hf = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
train_rows_all = _iter_conversation_turns(train_ds_hf, metadata_dict=metadata_dict, corpus_types=corpus_types)
train_rows_mined = [
    r for r in train_rows_all
    if (r.get('session_id'), r['track_id']) in train_mined_keys
][:DIAG_SAMPLE * 3]
import random
random.Random(42).shuffle(train_rows_mined)
_sample_s_pos(train_rows_mined, 'TRAIN-mined')

print(f'\n=== DEV sample (HF test split, unfiltered) ===')
dev_ds_hf = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
dev_rows = _iter_conversation_turns(dev_ds_hf, metadata_dict=metadata_dict, corpus_types=corpus_types)
_sample_s_pos(dev_rows, 'DEV (HF test)')

print('\n[diag] Interpretation:')
print('  - If TRAIN-mined mean s_pos is MUCH higher than DEV mean → mining filter')
print('    selection bias is real (the model only trains on "easy" queries).')
print('  - Switch MINING_STRATEGY = "simans" in cell 1 to eliminate the filter.')
print('  - SimANS keeps all queries; expected: train and dev distributions match.')
# Aggressive cleanup so cell 4's subprocess gets back the GPU memory.
# del every large tensor; gc; empty CUDA cache.
del model
try: del track_embs
except NameError: pass
try: del track_ids
except NameError: pass
try: del metadata_dict
except NameError: pass
import gc, torch; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


In [ ]:
# 4) Train + merge LoRA into base + push to Hub.
# Hub target: <HUB_REPO_MERGED> (gets overwritten each run).
# Open cell 5 (TensorBoard) in PARALLEL after this cell starts.
# Warm-start support: when RESUME_FROM is non-empty AND the path exists,
# pass --resume-from so training continues from that LoRA adapter.
# Optimizer/scheduler restart fresh (script-side intentional choice).
import os as _os_warmstart
_resume_flag = ''
if RESUME_FROM:
    if _os_warmstart.path.isdir(RESUME_FROM):
        _resume_flag = f'--resume-from {RESUME_FROM}'
        print(f'[train] WARM-START from {RESUME_FROM}')
    else:
        raise FileNotFoundError(
            f'RESUME_FROM is set to {RESUME_FROM!r} but the path does not exist. '
            f'Set RESUME_FROM = \'\' to train from scratch.'
        )
# Reproducibility: snapshot the full CONFIG + git SHA + start time to
# {TRAIN_OUTPUT_DIR}/config.json so months from now we know what produced
# this adapter. Done BEFORE training kicks off so even partial runs leave
# a config trace on Drive.
import json as _json_cfg, subprocess as _sp_cfg, time as _time_cfg, os as _os_cfg
_os_cfg.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
_cfg_snapshot = {
    'BRANCH': BRANCH, 'HUB_USER': HUB_USER, 'RUN_NAME': RUN_NAME, 'BGE_MODEL': BGE_MODEL,
    'HUB_REPO': HUB_REPO, 'HUB_REPO_MERGED': HUB_REPO_MERGED,
    'TRIPLES_JSONL': TRIPLES_JSONL, 'TRAIN_OUTPUT_DIR': TRAIN_OUTPUT_DIR,
    'SKIP_MINING': SKIP_MINING, 'RESUME_FROM': RESUME_FROM, 'SEED': SEED,
    'MAX_INPUT_ROWS': MAX_INPUT_ROWS, 'DEV_EVAL_ROWS': DEV_EVAL_ROWS,
    'PERCPOS_THRESHOLD': PERCPOS_THRESHOLD, 'POOL_SIZE': POOL_SIZE,
    'N_NEGATIVES': N_NEGATIVES, 'MINING_BATCH_SIZE': MINING_BATCH_SIZE,
    'EPOCHS': EPOCHS, 'PER_DEVICE_BATCH_SIZE': PER_DEVICE_BATCH_SIZE,
    'GRAD_ACCUM_STEPS': GRAD_ACCUM_STEPS, 'LR': LR, 'TEMPERATURE': TEMPERATURE,
    'LORA_RANK': LORA_RANK, 'LORA_ALPHA': LORA_ALPHA,
    'QUERY_MAX_LEN': QUERY_MAX_LEN, 'PASSAGE_MAX_LEN': PASSAGE_MAX_LEN,
    'SPLIT_KEY': SPLIT_KEY, 'VAL_FRACTION': VAL_FRACTION,
    'LOGGING_STEPS': LOGGING_STEPS, 'VAL_EVERY_N_STEPS': VAL_EVERY_N_STEPS,
    'VAL_FULL_CATALOG_EVERY_N': VAL_FULL_CATALOG_EVERY_N,
    'CHECKPOINT_EVERY_N_EPOCHS': CHECKPOINT_EVERY_N_EPOCHS,
    'GRADIENT_CHECKPOINTING': GRADIENT_CHECKPOINTING,
    'commit_sha': _sp_cfg.check_output(['git','-C','/content/recsys2026','rev-parse','HEAD']).decode().strip(),
    'started_at': _time_cfg.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
with open(f'{TRAIN_OUTPUT_DIR}/config.json', 'w') as _f_cfg:
    _json_cfg.dump(_cfg_snapshot, _f_cfg, indent=2)
print(f'[train] CONFIG snapshot -> {TRAIN_OUTPUT_DIR}/config.json (commit '
      f'{_cfg_snapshot["commit_sha"][:8]})')
_grad_ckpt_flag = '--gradient-checkpointing' if GRADIENT_CHECKPOINTING else '--no-gradient-checkpointing'
_ckpt_flag = f'--checkpoint-every-n-epochs {CHECKPOINT_EVERY_N_EPOCHS}'
!cd /content/recsys2026 && python -u scripts/train_bi_encoder.py \
    --triples {TRIPLES_JSONL} \
    --base-model {BGE_MODEL} \
    --output-dir {TRAIN_OUTPUT_DIR} \
    --hub-repo {HUB_REPO} \
    --results-dir {DRIVE_RESULTS} \
    --epochs {EPOCHS} \
    --per-device-batch-size {PER_DEVICE_BATCH_SIZE} \
    --gradient-accumulation-steps {GRAD_ACCUM_STEPS} \
    --lr {LR} \
    --seed {SEED} \
    --temperature {TEMPERATURE} \
    --lora-rank {LORA_RANK} \
    --lora-alpha {LORA_ALPHA} \
    --split-key {SPLIT_KEY} \
    --val-fraction {VAL_FRACTION} \
    --query-max-len {QUERY_MAX_LEN} \
    --passage-max-len {PASSAGE_MAX_LEN} \
    --logging-steps {LOGGING_STEPS} \
    --val-every-n-steps {VAL_EVERY_N_STEPS} \
    --val-full-catalog-every-n-steps {VAL_FULL_CATALOG_EVERY_N} \
    {_ckpt_flag} \
    {_resume_flag} \
    {_grad_ckpt_flag} \
    --merge --cleanup-after-push \
    2>&1 | tee {DRIVE_LOG_TRAIN}


In [ ]:
# 5) TensorBoard launcher — open in PARALLEL with cell 4.
# Compare for the leak check:
#   train/ndcg_inbatch  ← per-batch on training rows
#   val/ndcg            ← per-batch on held-out USER-DISJOINT val users
# Aligned curves (gap < ~0.05 sustained) → no leak.
# Also watch val/full_catalog_ndcg_at_20 — should climb above ~0.05.
%load_ext tensorboard
%tensorboard --logdir {TRAIN_OUTPUT_DIR}/runs --port={TENSORBOARD_PORT}


In [ ]:
# 6) Re-embed the ~47K-track catalog with the fine-tuned (merged) model.
#
# PATCH 4 (2026-05-22): catalog text now uses the SAME `id_to_metadata` format
# the training builder writes for pos/neg AND the format `chat_history_parser`
# uses to expand music-turn references in [HISTORY] blocks at inference. This
# closes the multi-way format asymmetry surfaced by the BlindA nDCG diagnosis.
# Old catalog vectors (built with format_track_text, 5 fields) are stale; this
# cell overwrites them with the aligned-format version.
import os, pickle, numpy as np, sys
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from build_bi_encoder_training_data import _format_history_music_turn

os.makedirs(CATALOG_OUT_DIR, exist_ok=True)
model = SentenceTransformer(HUB_REPO_MERGED, device='cuda')
model.max_seq_length = PASSAGE_MAX_LEN
model.tokenizer.truncation_side = 'right'
print(f'[catalog re-embed] max_seq_length={model.max_seq_length} truncation_side={model.tokenizer.truncation_side}')

tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
metadata_dict = {row['track_id']: dict(row) for row in tm}
corpus_types = ['track_name', 'artist_name', 'album_name']  # MUST match production config 021 + training builder
track_ids = [row['track_id'] for row in tm]
texts = [_format_history_music_turn(tid, metadata_dict, corpus_types) for tid in track_ids]
print(f'[catalog re-embed] format aligned with id_to_metadata; sample: {texts[0][:200]}')
embs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
embs = np.asarray(embs, dtype=np.float32)
out_path = os.path.join(CATALOG_OUT_DIR, 'track_embeddings.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'track_ids': track_ids, 'track_mat': embs}, f)
print(f'wrote {len(track_ids)} embeddings → {out_path}')


In [ ]:
# 7) Dev eval — uses the PRODUCTION code path (DENSE_LOCAL + build_retrieval_query
# + chat_history_parser-equivalent). Same path BlindA inference goes through, so
# the dev nDCG is apples-to-apples with the leaderboard's nDCG axis (modulo the
# rerank + responder stages, which apply equally to both).
#
# PATCH 2 (2026-05-22) + bugfix audit (2026-05-22):
# - Replaces the prior manual SentenceTransformer + cosine replica that had
#   subtle divergences from production.
# - FIX A: DENSE_LOCAL kwarg is `split_types`, not `track_split_types`.
# - FIX B: iterate USER turns and predict the NEXT music turn (mirrors
#   production's target_turn_number = user turn). Builds chat_history from
#   turns BEFORE the user turn — does NOT include the user turn itself —
#   so build_retrieval_query's last-user scan finds exactly one user_query
#   in session_memory (no duplicate).
import sys, math, os, json, time
import numpy as np
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.crs_baseline import build_retrieval_query
from mcrs.retrieval_modules.dense_local import DENSE_LOCAL
from mcrs.db_item.music_catalog import MusicCatalogDB

# Preflight: HF train↔test session-disjointness contract.
train_sess = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
test_sess  = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
train_sids = {s.get('session_id') for s in train_sess if s.get('session_id')}
test_sids  = {s.get('session_id') for s in test_sess  if s.get('session_id')}
sess_overlap = train_sids & test_sids
if sess_overlap:
    raise AssertionError(f'HF train/test share {len(sess_overlap)} session_ids — leak. Refusing to score.')
print(f'[preflight] session-disjointness OK (train={len(train_sids):,}, test={len(test_sids):,})')

# Build MusicCatalogDB exactly as production does.
item_db = MusicCatalogDB(
    dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
    split_types=['all_tracks'],
    corpus_types=['track_name', 'artist_name', 'album_name'],
)
print(f'[dev eval] item_db loaded: {len(item_db.metadata_dict):,} tracks')

# Build DENSE_LOCAL exactly as the wRRF factory does. Kwarg name fix: the
# constructor takes `split_types`, NOT `track_split_types`.
retriever = DENSE_LOCAL(
    dataset_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
    split_types=['all_tracks'],
    corpus_types=['track_name', 'artist_name', 'album_name'],
    cache_dir='/content/recsys2026/experiments/cache',
    model_name=HUB_REPO_MERGED,
    embed_label=EMBED_LABEL,
)
print(f'[dev eval] DENSE_LOCAL ready: encoder={HUB_REPO_MERGED}, embed_label={EMBED_LABEL}')

def _build_history_for_user_at(convs, user_pos):
    """Build chat_history exactly as production's chat_history_parser does:
    all turns BEFORE the target user turn (user_pos), with music turns
    expanded via item_db.id_to_metadata. Does NOT include the user turn
    itself — that goes into session_memory separately as the appended
    {role:user, content:user_query} entry. This mirrors batch_chat which
    receives chat_history (no user_query) then appends user_query."""
    chat_history = []
    for prev in convs[:user_pos]:
        role = prev.get('role')
        content = prev.get('content') or ''
        if role == 'user':
            chat_history.append({'role': 'user', 'content': content})
        elif role == 'music':
            expanded = item_db.id_to_metadata(content) if content in item_db.metadata_dict else content
            chat_history.append({'role': 'assistant', 'content': expanded})
        elif role == 'assistant':
            chat_history.append({'role': 'assistant', 'content': content})
    return chat_history

# Walk test sessions. For each USER turn whose NEXT turn is a MUSIC turn,
# emit one (query, gold) pair. This matches the (user_query, target track)
# semantics that BlindA's target_turn_number marks.
queries_text, gold_tids = [], []
for sess in test_sess:
    convs = sess.get('conversations', [])
    for user_pos, turn in enumerate(convs):
        if turn.get('role') != 'user':
            continue
        # Need a music turn immediately after to have a gold.
        if user_pos + 1 >= len(convs) or convs[user_pos + 1].get('role') != 'music':
            continue
        user_query = turn.get('content') or ''
        gold = convs[user_pos + 1].get('content')
        if not user_query or not gold:
            continue
        history = _build_history_for_user_at(convs, user_pos)
        # session_memory = chat_history + appended user_query. This is what
        # batch_chat does at production (crs_baseline.py:445-449).
        session_memory = history + [{'role': 'user', 'content': user_query}]
        cg = sess.get('conversation_goal') or {}
        goal_text = (cg.get('listener_goal') or '').strip() or None
        qtext = build_retrieval_query(
            session_memory,
            mode='bge_m3_structured',
            goal_text=goal_text,
            user_profile=sess.get('user_profile'),
        )
        queries_text.append(qtext)
        gold_tids.append(gold)
        if len(queries_text) >= DEV_EVAL_ROWS:
            break
    if len(queries_text) >= DEV_EVAL_ROWS:
        break

print(f'[dev eval] {len(queries_text)} queries built via production code path')
print(f'[dev eval] sample query[:400]:\n  {queries_text[0][:400]}')

# Score via DENSE_LOCAL.
top20 = retriever.batch_text_to_item_retrieval(queries_text, topk=20)
ndcgs = []
for gold, ranked in zip(gold_tids, top20):
    if gold in ranked:
        rank = ranked.index(gold) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / max(1, len(ndcgs)))
print(f'\n[dev eval] standalone DENSE_LOCAL nDCG@20 = {mean_ndcg:.4f} (n={len(ndcgs)})')
print(f'[dev eval] this number is APPLES-TO-APPLES with the production code path.')

# Persist.
_dev_eval = {
    'run_name': RUN_NAME, 'hub_repo': HUB_REPO_MERGED,
    'epochs': EPOCHS, 'seed': SEED,
    'mining_strategy': MINING_STRATEGY,
    'dev_eval_rows': len(ndcgs),
    'standalone_ndcg_at_20': mean_ndcg,
    'eval_path': 'DENSE_LOCAL (production code path)',
    'completed_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
with open(f'{TRAIN_OUTPUT_DIR}/dev_eval.json', 'w') as f:
    json.dump(_dev_eval, f, indent=2)
print(f'[dev eval] result → {TRAIN_OUTPUT_DIR}/dev_eval.json')
